# 02 — A0 generation (no attack), all model runs x test sets (PLAN.md §5, §13.4)
8 runs (BASE, SDT, GC, IL, RL, RL_noguard, ST, SP) x {harmful_test, xstest_test, mmlu_test} = 24 jobs,
spread over both T4s (one worker process per GPU). fp16, batched greedy decoding.

Outputs: `results/gen/<run>__A0__<split>.parquet` + `.json` manifest, pushed after each job to the
**private** HF dataset `bennytimz/aftermarket-runs` (namespace read from `whoami()`). Re-running skips finished jobs, so after a
timeout just run the notebook again.

Requires: Kaggle secret `HF_TOKEN` with **write** access, accelerator **GPU T4 x2**.

**Test-set rule:** these are leaderboard cells (others' models, no attack), so they run on the test splits with `--final`: no choices are made from them. All tuning (attacks, AM, thresholds) uses dev splits only.

**Resume:** the runner first pulls finished jobs from the private Hub repo, so after a dead session just commit again; only missing jobs run.

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import subprocess, yaml
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
cfg = yaml.safe_load(open("configs/gen_a0.yaml"))
print({k: cfg[k] for k in ["runs", "splits", "precision", "batch_size", "gpus"]})

In [ ]:
# Run all pending jobs on both GPUs (streams worker logs; also in results/gen/logs/)
!PYTHONPATH=src python -m aftermarket.generate --config configs/gen_a0.yaml --final

In [ ]:
# Final summary (always runs; stats only, no completion text)
import glob, json
import pandas as pd
try:
    recs = []
    for p in sorted(glob.glob("results/gen/*.json")):
        m = json.load(open(p))
        j = m["job"]
        recs.append({"run": j["run_id"], "attack": j["attack_id"], "split": j["split"], "n": m["n_prompts"],
                     "prompts_per_sec": m["prompts_per_sec"], "tok_per_sec": m["new_tokens_per_sec"],
                     "pushed": m.get("pushed")})
    summary = pd.DataFrame(recs)
    summary = summary[summary.attack == "A0"] if len(summary) else summary
    print(summary.to_string(index=False) if len(summary) else "no finished jobs")
    expected = {(r, s) for r in cfg["runs"] for s in cfg["splits"]}
    done = set(zip(summary.get("run", []), summary.get("split", [])))
    print(f"{len(done & expected)}/{len(expected)} A0 jobs done; missing: {sorted(expected - done)}")
except Exception as e:
    print(f"summary failed: {type(e).__name__}: {e}")